# Imports básicos y configuración

In [4]:
# Cell 1 – Imports y setup (Notebook 2)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q torchmetrics torchvision matplotlib pandas opencv-python
else:
    %pip install -q opencv-python

import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.transforms import functional as TF

import matplotlib.pyplot as plt
import cv2

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running in Colab:", IN_COLAB)
print("Using device:", device)
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Running in Colab: False
Using device: cuda
Torch: 2.9.1+cu128
Torchvision: 0.24.1+cu128


# Paths y comprobación de dataset YOLO existente

In [5]:
# Cell 2 – Paths: reutilizar EXACTO dataset del Notebook 1

if IN_COLAB:
    PROJ_ROOT = Path("/content/hand_bbox_baseline")
else:
    PROJ_ROOT = Path("/home/fran/Universidad/TFG/hand_bbox_baseline")

YOLO_ROOT = PROJ_ROOT / "data" / "hands_yolo"

print("PROJ_ROOT:", PROJ_ROOT)
print("YOLO_ROOT:", YOLO_ROOT)

# Comprobar cuántos ficheros hay en cada split (debería ser 4178 / 522 / 523)
for split in ["train", "val", "test"]:
    img_dir = YOLO_ROOT / "images" / split
    lbl_dir = YOLO_ROOT / "labels" / split

    num_imgs = (
        len(list(img_dir.glob("*.jpg")))
        + len(list(img_dir.glob("*.png")))
        + len(list(img_dir.glob("*.jpeg")))
    )
    num_lbls = len(list(lbl_dir.glob("*.txt")))

    print(f"{split}: imgs={num_imgs}, labels={num_lbls}")


PROJ_ROOT: /home/fran/Universidad/TFG/hand_bbox_baseline
YOLO_ROOT: /home/fran/Universidad/TFG/hand_bbox_baseline/data/hands_yolo
train: imgs=4178, labels=4178
val: imgs=522, labels=522
test: imgs=523, labels=523


# Dataset HandsYOLODataset + DataLoaders

In [6]:
# Cell 3 – Dataset class and DataLoaders

from PIL import Image

class HandsYOLODataset(Dataset):
    """
    Lee imágenes + etiquetas en formato YOLO (txt) y devuelve:
    - image: tensor [3,H,W] en [0,1]
    - target: dict con 'boxes' (N,4) en pixeles y 'labels' (N,)
    """
    def __init__(self, img_dir, lbl_dir, transforms=None):
        self.img_dir = Path(img_dir)
        self.lbl_dir = Path(lbl_dir)
        self.transforms = transforms

        self.img_paths = (
            sorted(self.img_dir.glob("*.jpg")) +
            sorted(self.img_dir.glob("*.jpeg")) +
            sorted(self.img_dir.glob("*.png"))
        )
        if len(self.img_paths) == 0:
            print("⚠️ WARNING: no images found in", self.img_dir)

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = Image.open(img_path).convert("RGB")
        w, h = image.size

        label_path = self.lbl_dir / (img_path.stem + ".txt")
        boxes = []
        labels = []

        if label_path.exists():
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls, cx, cy, bw, bh = parts
                    cx = float(cx); cy = float(cy)
                    bw = float(bw); bh = float(bh)

                    x1 = (cx - bw/2.0) * w
                    y1 = (cy - bh/2.0) * h
                    x2 = (cx + bw/2.0) * w
                    y2 = (cy + bh/2.0) * h

                    boxes.append([x1, y1, x2, y2])
                    # solo una clase "hand" → id 1 (0 se usa para background en TorchVision)
                    labels.append(1)

        if len(boxes) > 0:
            boxes  = torch.tensor(boxes,  dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
        else:
            # imagen sin manos (por seguridad)
            boxes  = torch.zeros((0,4), dtype=torch.float32)
            labels = torch.zeros((0,),  dtype=torch.int64)

        if self.transforms is not None:
            image = self.transforms(image)
        else:
            image = F.to_tensor(image)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
        }
        return image, target, str(img_path)

def collate_fn(batch):
    # batch: list of (image, target, path)
    images, targets, paths = zip(*batch)
    return list(images), list(targets), list(paths)

BATCH_SIZE = 4

train_dataset = HandsYOLODataset(train_img_dir, train_lbl_dir)
val_dataset   = HandsYOLODataset(val_img_dir,   val_lbl_dir)
test_dataset  = HandsYOLODataset(test_img_dir,  test_lbl_dir)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=2, pin_memory=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=2, pin_memory=True
)

print("Train samples:", len(train_dataset))
print("Val samples:",   len(val_dataset))
print("Test samples:",  len(test_dataset))


NameError: name 'train_img_dir' is not defined

# Visual sanity check (como en el otro notebook)

In [ ]:
# Cell 4 – Visual sanity check

def show_batch(dataset, n=3, split_name="train"):
    import random
    idxs = random.sample(range(len(dataset)), min(n, len(dataset)))
    for i in idxs:
        img, target, path = dataset[i]
        img_np = img.permute(1,2,0).numpy()
        h, w, _ = img_np.shape

        fig, ax = plt.subplots(1,1, figsize=(6,6))
        ax.imshow(img_np)
        for (x1,y1,x2,y2) in target["boxes"]:
            rect = plt.Rectangle((x1,y1), x2-x1, y2-y1,
                                 fill=False, edgecolor="lime", linewidth=2)
            ax.add_patch(rect)
        ax.set_title(f"{split_name.upper()} - {Path(path).name} (#{len(target['boxes'])} boxes)")
        ax.axis("off")
        plt.show()

show_batch(train_dataset, n=3, split_name="train")
show_batch(val_dataset,   n=3, split_name="val")


# Definir modelos (SSDLite y Faster R-CNN) + helper constructor

In [ ]:
# Cell 5 – Model builders (SSDLite and Faster R-CNN)

from torchvision.models.detection import (
    ssdlite320_mobilenet_v3_large,
    fasterrcnn_mobilenet_v3_large_fpn,
)
from torchvision.models.detection.ssdlite import SSDLiteClassificationHead
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

NUM_CLASSES = 2  # background + hand

def create_ssdlite_mbv3(num_classes=NUM_CLASSES):
    # Cargamos pesos COCO y cambiamos el head de clasificación
    weights = torchvision.models.detection.SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
    model = ssdlite320_mobilenet_v3_large(weights=weights)
    model.eval()  # para congelar stats inicialmente

    in_channels = model.head.classification_head.conv[0].in_channels
    num_anchors = model.head.classification_head.num_anchors
    model.head.classification_head = SSDLiteClassificationHead(
        in_channels, num_anchors, num_classes
    )
    return model

def create_fasterrcnn_mbv3(num_classes=NUM_CLASSES):
    weights = torchvision.models.detection.FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
    model = fasterrcnn_mobilenet_v3_large_fpn(weights=weights)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


# Training loop genérico para detectores TorchVision

In [ ]:
# Cell 6 – Generic training loop for detection models

def train_detector(model, train_loader, val_loader, epochs=20, lr=1e-3, exp_name="exp"):
    model = model.to(device)
    model.train()

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr)

    history = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        n_batches = 0

        for images, targets, paths in train_loader:
            images = [img.to(device) for img in images]
            tgts = []
            for t in targets:
                t = {k: v.to(device) for k, v in t.items()}
                tgts.append(t)

            loss_dict = model(images, tgts)
            loss = sum(loss_dict.values())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            n_batches += 1

        train_loss = running_loss / max(1, n_batches)

        # Pequeño eval de validación por pérdida (usamos unas pocas batches para no morir)
        model.eval()
        val_loss = 0.0
        val_batches = 0
        with torch.no_grad():
            for i, (images, targets, paths) in enumerate(val_loader):
                if i >= 10:  # limitar para que vaya fluido
                    break
                images = [img.to(device) for img in images]
                tgts = [{k: v.to(device) for k, v in t.items()} for t in targets]
                loss_dict = model(images, tgts)
                loss = sum(loss_dict.values())
                val_loss += loss.item()
                val_batches += 1

        val_loss = val_loss / max(1, val_batches)

        print(f"[{exp_name}] Epoch {epoch+1}/{epochs}  "
              f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
        })

    hist_df = pd.DataFrame(history)
    out_dir = PROJ_ROOT / "runs_torchvision"
    out_dir.mkdir(parents=True, exist_ok=True)
    hist_path = out_dir / f"{exp_name}_loss_history.csv"
    hist_df.to_csv(hist_path, index=False)
    print("Saved loss history to:", hist_path)

    return model, hist_df


# Entrenar SSDLite y Faster R-CNN

In [ ]:
# Cell 7 – Train SSDLite and Faster R-CNN

EPOCHS = 100  # ajusta según el tiempo que tengas
LR_SSD = 1e-3
LR_FRCNN = 5e-4

experiments = {}
histories = {}

# SSDLite
ssdlite_model = create_ssdlite_mbv3()
print("\n=== Training SSDLite320-MobileNetV3 ===")
ssdlite_model, hist_ssd = train_detector(
    ssdlite_model, train_loader, val_loader,
    epochs=EPOCHS, lr=LR_SSD, exp_name="ssdlite_mbv3"
)
experiments["ssdlite_mbv3"] = ssdlite_model
histories["ssdlite_mbv3"] = hist_ssd

# Faster R-CNN
faster_model = create_fasterrcnn_mbv3()
print("\n=== Training FasterRCNN-MobileNetV3-FPN ===")
faster_model, hist_frcnn = train_detector(
    faster_model, train_loader, val_loader,
    epochs=EPOCHS, lr=LR_FRCNN, exp_name="fasterrcnn_mbv3"
)
experiments["fasterrcnn_mbv3"] = faster_model
histories["fasterrcnn_mbv3"] = hist_frcnn


# Plot de curvas de pérdida

In [ ]:
# Cell 8 – Plot loss curves

def plot_loss(hist_df, title):
    plt.figure(figsize=(8,5))
    plt.plot(hist_df["epoch"], hist_df["train_loss"], label="train loss")
    plt.plot(hist_df["epoch"], hist_df["val_loss"],   label="val loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

for name, hist in histories.items():
    print(f"\n=== Loss curves for {name} ===")
    plot_loss(hist, name)


# Utilidades de boxes e IoU (como en Notebook 01)

In [ ]:
# Cell 9 – Box utilities and IoU (same spirit as YOLO notebook)

def yolo_xywhn_to_xyxyn(cx, cy, w, h):
    x1 = cx - w / 2.0
    y1 = cy - h / 2.0
    x2 = cx + w / 2.0
    y2 = cy + h / 2.0
    return np.array([x1, y1, x2, y2], dtype=np.float32)

def box_iou_xyxy(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h

    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])
    union = area_a + area_b - inter
    if union <= 0.0:
        return 0.0
    return inter / union

def greedy_match_boxes(pred_boxes, gt_boxes, iou_min=0.1):
    matches = []
    if len(pred_boxes) == 0 or len(gt_boxes) == 0:
        return matches

    pred_used = set()
    gt_used = set()

    iou_matrix = np.zeros((len(pred_boxes), len(gt_boxes)), dtype=np.float32)
    for i, pb in enumerate(pred_boxes):
        for j, gb in enumerate(gt_boxes):
            iou_matrix[i, j] = box_iou_xyxy(pb, gb)

    while True:
        i, j = np.unravel_index(np.argmax(iou_matrix), iou_matrix.shape)
        best_iou = iou_matrix[i, j]
        if best_iou < iou_min:
            break
        if i in pred_used or j in gt_used:
            iou_matrix[i, j] = -1.0
            continue

        matches.append((i, j, best_iou))
        pred_used.add(i)
        gt_used.add(j)
        iou_matrix[i, :] = -1.0
        iou_matrix[:, j] = -1.0

    return matches


# Evaluar métricas extra (IoU, MAE, MSE, R²) en validación

In [ ]:
# Cell 10 – Extra metrics evaluation on val set

def evaluate_extra_metrics_torchvision(model, dataset, device=device,
                                       conf_threshold=0.25, max_images=None):
    model = model.to(device)
    model.eval()

    all_gt_coords = []
    all_pred_coords = []
    all_ious = []

    with torch.no_grad():
        for idx in range(len(dataset)):
            if max_images is not None and idx >= max_images:
                break

            img, target, path = dataset[idx]
            w, h = img.shape[2], img.shape[1]  # [C,H,W]

            img_dev = img.to(device)
            outputs = model([img_dev])[0]

            pred_boxes = outputs["boxes"].cpu().numpy()
            scores     = outputs["scores"].cpu().numpy()

            # Filtramos por confianza
            keep = scores >= conf_threshold
            pred_boxes = pred_boxes[keep]

            # Normalizamos a [0,1]
            pred_boxes_norm = []
            for bx in pred_boxes:
                x1,y1,x2,y2 = bx
                pred_boxes_norm.append(
                    np.array([x1/w, y1/h, x2/w, y2/h], dtype=np.float32)
                )

            gt_boxes_px = target["boxes"].numpy()
            gt_boxes_norm = []
            for bx in gt_boxes_px:
                x1,y1,x2,y2 = bx
                gt_boxes_norm.append(
                    np.array([x1/w, y1/h, x2/w, y2/h], dtype=np.float32)
                )

            matches = greedy_match_boxes(pred_boxes_norm, gt_boxes_norm, iou_min=0.1)

            for (pi, gi, iou) in matches:
                pb = pred_boxes_norm[pi]
                gb = gt_boxes_norm[gi]
                all_ious.append(iou)
                all_pred_coords.append(pb)
                all_gt_coords.append(gb)

    if len(all_gt_coords) == 0:
        print("No matches found; check thresholds.")
        return None

    all_gt = np.stack(all_gt_coords, axis=0)
    all_pred = np.stack(all_pred_coords, axis=0)
    all_ious = np.array(all_ious)

    mae = np.mean(np.abs(all_pred - all_gt))
    mse = np.mean((all_pred - all_gt)**2)

    y_true = all_gt.reshape(-1)
    y_pred = all_pred.reshape(-1)
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")

    metrics = {
        "mean_iou": float(np.mean(all_ious)),
        "mae": float(mae),
        "mse": float(mse),
        "r2": float(r2),
        "num_matches": len(all_gt_coords),
    }
    return metrics

extra_metrics_02 = {}
for name, model in experiments.items():
    print(f"\n=== Extra metrics (val) for {name} ===")
    m = evaluate_extra_metrics_torchvision(model, val_dataset, conf_threshold=0.25, max_images=None)
    print(m)
    extra_metrics_02[name] = m


# Parámetros, FLOPs y latencias

In [ ]:
# Cell 11 – Params, FLOPs (optional) and latency

IMG_SIZE = 416  # solo para medir latencia/FLOPs

def count_params_generic(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def compute_flops_generic(model, img_size=IMG_SIZE):
    # muchos modelos de detección no funcionan bien con thop -> lo intentamos y si falla devolvemos None
    dummy = torch.randn(1, 3, img_size, img_size).to(device)
    # TorchVision detection espera lista de imágenes
    inp = [dummy[0]]
    try:
        flops, params = profile(model, inputs=(inp,), verbose=False)
        return flops, params
    except Exception as e:
        print("FLOPs estimation failed:", e)
        return None, None

def benchmark_latency_generic(model, img_size=IMG_SIZE, device_str="cuda",
                              n_warmup=10, n_iters=50):
    dev = torch.device(device_str)
    model = model.to(dev)
    model.eval()

    dummy = torch.randn(1, 3, img_size, img_size, device=dev)
    imgs = [dummy[0]]

    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(imgs)
        if device_str == "cuda":
            torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(n_iters):
            t0 = time.perf_counter()
            _ = model(imgs)
            if device_str == "cuda":
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            times.append(t1 - t0)

    return np.mean(times) * 1000.0  # ms

model_stats_02 = {}

for name, model in experiments.items():
    print(f"\n=== Model stats for {name} ===")
    total_params, trainable_params = count_params_generic(model)
    params_m = total_params / 1e6
    print(f"Params: {params_m:.3f} M ({total_params:,} total)")
    print(f"Trainable params: {trainable_params:,}")

    flops, params_thop = compute_flops_generic(model, IMG_SIZE)
    if flops is not None:
        flops_g = flops / 1e9
        print(f"Approx MACs: {flops_g:.3f} G")
    else:
        flops_g = None

    gpu_ms = None
    if torch.cuda.is_available():
        gpu_ms = benchmark_latency_generic(model, IMG_SIZE, device_str="cuda")
        print(f"GPU latency: {gpu_ms:.2f} ms / img")

    cpu_ms = benchmark_latency_generic(model, IMG_SIZE, device_str="cpu")
    print(f"CPU latency: {cpu_ms:.2f} ms / img")

    model_stats_02[name] = {
        "params_m": params_m,
        "flops_gmacs": flops_g,
        "gpu_ms": gpu_ms,
        "cpu_ms": cpu_ms,
    }


# Tabla comparativa y guardar CSV

In [ ]:
# Cell 12 – Build comparison table and save

rows = []

for name in experiments.keys():
    stats = model_stats_02.get(name, {})
    extra = extra_metrics_02.get(name, {})

    row = {
        "experiment": name,
        "params_M": stats.get("params_m"),
        "FLOPs_GMACs": stats.get("flops_gmacs"),
        "GPU_ms_per_img": stats.get("gpu_ms"),
        "CPU_ms_per_img": stats.get("cpu_ms"),
        "mean_IoU": extra.get("mean_iou"),
        "MAE_coords": extra.get("mae"),
        "MSE_coords": extra.get("mse"),
        "R2_coords": extra.get("r2"),
        "num_matches": extra.get("num_matches"),
    }
    rows.append(row)

df_summary_02 = pd.DataFrame(rows)
df_summary_02


In [ ]:
# Still Cell 12 – Save to CSV

summary_path_02 = PROJ_ROOT / "hands_model_comparison_torchvision.csv"
df_summary_02.to_csv(summary_path_02, index=False)
print("Saved comparison table to:", summary_path_02)


# Inferencia rápida y visualización con cada modelo

In [ ]:
# Cell 13 – Inference demo for each model on a custom image

import cv2

def demo_inference_torchvision(model, img_path, title="demo", score_thr=0.4):
    model = model.to(device)
    model.eval()

    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        print("No se pudo leer la imagen:", img_path)
        return
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)

    img_tensor = F.to_tensor(img_pil).to(device)
    with torch.no_grad():
        out = model([img_tensor])[0]

    boxes = out["boxes"].cpu().numpy()
    scores = out["scores"].cpu().numpy()

    img_draw = img_rgb.copy()
    h, w, _ = img_draw.shape

    for bx, sc in zip(boxes, scores):
        if sc < score_thr:
            continue
        x1,y1,x2,y2 = bx.astype(int)
        cv2.rectangle(img_draw, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(img_draw, f"hand {sc:.2f}", (x1, max(0, y1-5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2, cv2.LINE_AA)

    plt.figure(figsize=(6,6))
    plt.imshow(img_draw)
    plt.title(title)
    plt.axis("off")
    plt.show()

# Ejemplo de uso (cambia la ruta por una foto tuya de pruebas):
test_img_path = Path("fototest3.jpeg")  # o cualquier foto que quieras

if test_img_path.exists():
    demo_inference_torchvision(experiments["ssdlite_mbv3"], test_img_path, title="SSDLite-MBV3")
    demo_inference_torchvision(experiments["fasterrcnn_mbv3"], test_img_path, title="FasterRCNN-MBV3")
else:
    print("⚠️ No existe la imagen de demo, cambia 'test_img_path'.")
